# Convert to TFLite – Light-Level Classifier

**Author:** suriyakumar P

Train a light-level classifier for **Dark, Normal, Bright**, convert it to `.tflite`, and verify inference in Python.

In [ ]:
import numpy as np
import tensorflow as tf

print('TensorFlow version:', tf.__version__)

## 1. Create Light-Level Data

Categories:
- **Dark:** below 300
- **Normal:** 300 to 699
- **Bright:** 700 and above

In [ ]:
light_levels = np.array([50,100,150,200,250,300,400,500,600,650,699,700,800,900,1000], dtype=np.float32)
labels = np.array([0,0,0,0,0,1,1,1,1,1,1,2,2,2,2], dtype=np.int32)

x_train = light_levels.reshape(-1, 1) / 1000.0
class_names = ['Dark', 'Normal', 'Bright']

print('Light levels:', light_levels)
print('Labels:', [class_names[i] for i in labels])

## 2. Build and Train the Classifier

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.02), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, labels, epochs=300, verbose=0)

loss, accuracy = model.evaluate(x_train, labels, verbose=0)
print(f'Training accuracy: {accuracy * 100:.2f}%')

## 3. Convert to TensorFlow Lite

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_path = 'light_level_classifier.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print('Created:', tflite_path)
print('Model size:', len(tflite_model), 'bytes')

## 4. Verify TFLite Inference in Python

In [ ]:
interpreter = tf.lite.Interpreter(model_path='light_level_classifier.tflite')
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

test_values = [100, 500, 900]
print('TFLite inference results:')

for value in test_values:
    input_data = np.array([[value / 1000.0]], dtype=np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])[0]
    prediction = int(np.argmax(output))
    print(f'{value} -> {class_names[prediction]} (confidence: {output[prediction]:.3f})')

## Result

The trained TensorFlow model is converted into `light_level_classifier.tflite`. The TFLite interpreter is used directly in Python to verify predictions for Dark, Normal, and Bright light levels.

Expected test results:
- `100` → Dark
- `500` → Normal
- `900` → Bright